### Sara's data exploration for MiBici

This notebook explores the current MiBici data to come to a decision on the initial data transformations that need to be done in order to answer P1, P2 and P3 individually

In [1]:
import pandas as pd


In [2]:
stations_df = pd.read_csv('../data/raw/nomenclatura_2026_09.csv')
stations_df.head()

,id,name,latitude,longitude,dpcapacity
0,2,(GDL-001) C. Epigmenio Glez./ Av. 16 de Sept.,20.666378,-103.348820,15
1,3,(GDL-002) C. Colonias / Av. Niños héroes,20.667228,-103.366000,15
2,4,(GDL-003) C. Vidrio / Av. Chapultepec,20.667690,-103.368252,19
3,5,(GDL-004) C. Ghilardi /C. Miraflores,20.691847,-103.362549,19
4,6,(GDL-005) C. San Diego /Calzada Independencia,20.681158,-103.339363,11


In [3]:
sep_2026_data = pd.read_csv('../data/raw/datos_abiertos_2026_09.csv')
sep_2026_data.head()

,Viaje_Id,Usuario_Id,Genero,Año_de_nacimiento,Inicio_del_viaje,Fin_del_viaje,Origen_Id,Destino_Id
0,45705294,4722545,M,1997.0,2026-09-01 00:00:59,2026-09-01 00:18:55,283,49
1,45705295,2344334,M,2004.0,2026-09-01 00:01:36,2026-09-01 00:17:52,67,383
2,45705301,4454420,M,2001.0,2026-09-01 00:03:36,2026-09-01 00:19:31,395,357
3,45705303,4512715,M,1996.0,2026-09-01 00:03:59,2026-09-01 00:07:07,371,57
4,45705304,2765763,M,2002.0,2026-09-01 00:04:09,2026-09-01 00:30:19,357,80


### Deleting unnecessary columns for analysis
** Note this should be the first step in our shared pipeline

In [4]:
sep_2026_data.drop(columns=['Usuario_Id', 'Genero', 'Año_de_nacimiento'], inplace=True)

## Analyzing datasets individually

### Stations

In [5]:
stations_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 484 entries, 0 to 483
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   id          484 non-null    int64  
 1   name        484 non-null    str    
 2   latitude    484 non-null    float64
 3   longitude   484 non-null    float64
 4   dpcapacity  484 non-null    int64  
dtypes: float64(2), int64(2), str(1)
memory usage: 38.9 KB


Out of 484 we have unique values for id and name which is expected, latitude and longitude should be analyzed together in order to see if there are any repeated locations.

In [6]:
stations_df.nunique()

id            484
name          484
latitude      480
longitude     481
dpcapacity      9
dtype: int64

It looks like some location pairs are repeated under different name and ids, upon further investigation, I found out the ex-(ZPN-026) and ex-(ZPN-044) are old stations that were renamed to (GDL-180) and (GDL-181) respectively. So the second step in our pipeline should be to deprecate the old names in both datasets. We'll do this by replacing any id 116 with 248, and any id 138 with 249. An automated extraction of these deprectated stations should be considered in the future.

In [7]:
pair_counts = stations_df.groupby(['latitude', 'longitude'])['latitude'].transform('size')
stations_df[pair_counts > 1]

,id,name,latitude,longitude,dpcapacity
111,116,ex-(ZPN-026) C. 16 Sep. / C. Fray A. de Segovia,20.68692,-103.33467,0
129,138,ex-(ZPN-044),20.69705,-103.36322,0
238,248,(GDL-180) C. Montes Pirineos/Salvador Quevedo,20.68692,-103.33467,19
239,249,(GDL-181) C.Ignacio Rmrz / Av Plan de San Lui,20.69705,-103.36322,15


In [8]:
replacement_stations = {116: 248, 138: 249}

In [9]:
stations_df[stations_df.isna().any(axis=1)]

,id,name,latitude,longitude,dpcapacity


In [10]:
stations_df['dpcapacity'].describe()

count    484.000000
mean      16.882231
std        4.126942
min        0.000000
25%       15.000000
50%       15.000000
75%       19.000000
max       39.000000
Name: dpcapacity, dtype: float64

### Monthly data

Initial observation: timestamps are of type str

In [11]:
sep_2026_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 490056 entries, 0 to 490055
Data columns (total 5 columns):
 #   Column            Non-Null Count   Dtype
---  ------            --------------   -----
 0   Viaje_Id          490056 non-null  int64
 1   Inicio_del_viaje  490056 non-null  str  
 2   Fin_del_viaje     490056 non-null  str  
 3   Origen_Id         490056 non-null  int64
 4   Destino_Id        490056 non-null  int64
dtypes: int64(3), str(2)
memory usage: 36.5 MB


There aren't any null entries

In [12]:
sep_2026_data[sep_2026_data.isna().any(axis=1)]

,Viaje_Id,Inicio_del_viaje,Fin_del_viaje,Origen_Id,Destino_Id


It seems like the `Viaje_Id` column doesn't match the quantity of records 489298 != 490056, this equals to 758 duplicated trips

In [13]:
print(sep_2026_data.shape)
sep_2026_data.nunique()

(490056, 5)


Viaje_Id            489298
Inicio_del_viaje    420403
Fin_del_viaje       421749
Origen_Id              468
Destino_Id             468
dtype: int64

In [14]:
sep_2026_data.drop_duplicates(inplace=True)


Now we transform the timestamp columns into a single minute duration column

In [15]:
sep_2026_data['Inicio_del_viaje'] = pd.to_datetime(sep_2026_data['Inicio_del_viaje'])
sep_2026_data['Fin_del_viaje'] = pd.to_datetime(sep_2026_data['Fin_del_viaje'])

sep_2026_data['duration'] = sep_2026_data['Fin_del_viaje'] - sep_2026_data['Inicio_del_viaje']
sep_2026_data['Duracion_Minutos'] = sep_2026_data['duration'].dt.total_seconds() / 60
sep_2026_data.drop(columns= 'duration', inplace=True)

sep_2026_data


,Viaje_Id,Inicio_del_viaje,Fin_del_viaje,Origen_Id,Destino_Id,Duracion_Minutos
0,45705294,2026-09-01 00:00:59,2026-09-01 00:18:55,283,49,17.933333
1,45705295,2026-09-01 00:01:36,2026-09-01 00:17:52,67,383,16.266667
2,45705301,2026-09-01 00:03:36,2026-09-01 00:19:31,395,357,15.916667
3,45705303,2026-09-01 00:03:59,2026-09-01 00:07:07,371,57,3.133333
4,45705304,2026-09-01 00:04:09,2026-09-01 00:30:19,357,80,26.166667
...,...,...,...,...,...,...
490050,46255553,2026-09-30 23:56:38,2026-10-01 00:14:52,73,73,18.233333
490051,46255554,2026-09-30 23:57:23,2026-09-30 23:59:46,43,44,2.383333
490052,46255556,2026-09-30 23:59:13,2026-10-01 00:04:17,210,413,5.066667
490054,46255557,2026-09-30 23:59:32,2026-10-01 00:06:16,157,271,6.733333


### Replacing deprecated stations

In [16]:
replacement_stations

{116: 248, 138: 249}

Since the reclassification of station was done in 2019, the newer monthly entries do not contain the old reference. This step will be considered in the pipeline for now in the case any data from those years is ever used. 

In [17]:
mask = sep_2026_data['Origen_Id'].isin(replacement_stations.keys()) | sep_2026_data['Destino_Id'].isin(replacement_stations.keys())

sep_2026_data[mask]


,Viaje_Id,Inicio_del_viaje,Fin_del_viaje,Origen_Id,Destino_Id,Duracion_Minutos


In [18]:
target_cols = ['Origen_Id', 'Destino_Id']
sep_2026_data[target_cols] = sep_2026_data[target_cols].replace(replacement_stations)

In [19]:
stations_df = stations_df[~stations_df[['id']].isin(replacement_stations.keys()).any(axis=1)]
stations_df.shape

(482, 5)

### Recommended steps to be done in the initial separate pipeline
1. Drop 'Usuario_Id', 'Genero', 'Año_de_nacimiento' from monthly data sets
2. Drop duplicates and null value columns (if any are found) from monthly data sets
3. Transform the timestamp columns into a single minute duration column from monthly data sets and keep the start and end dates
4. Drop old duration columns
5. Replace deprecated stations from monthly data sets
6. Remove deprecated stations from stations df
7. Join DFs